# 🚀 Milestone 5 — Fine-Tuning Pretrained Audio Transformer (AST)
**This is your key to hitting 0.80+ Macro F1**

### What is AST?
- **Audio Spectrogram Transformer** — pretrained on AudioSet (2M audio clips)
- Already knows what drums, bass, vocals, harmony sound like
- We just teach it the 10 genre labels = **fine-tuning**
- Think of it like hiring an expert musician and giving them a short test

### Why this works:
- ImageNet pretrained CNNs work well on new vision tasks
- AudioSet pretrained transformers work well on new audio tasks
- Much less data needed, much faster convergence

In [ ]:
!pip install transformers librosa wandb -q

In [ ]:
import os, random, warnings, time
import numpy as np
import pandas as pd
import librosa
import wandb
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import ASTFeatureExtractor, ASTForAudioClassification
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Device: {DEVICE}')
if DEVICE.type != 'cuda':
    print('⚠️  GPU not found! AST fine-tuning will be SLOW without GPU.')
    print('   Enable GPU: Settings → Accelerator → GPU T4 x2')

BASE        = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'
STEMS_DIR   = f'{BASE}/genres_stems'
MASHUPS_DIR = f'{BASE}/mashups'
TEST_CSV    = f'{BASE}/test.csv'
GENRES      = ['blues','classical','country','disco','hiphop','jazz','metal','pop','reggae','rock']
ROLL_NO     = 'YOUR_ROLL_NO'  # ⚠️ change this!
USE_STEMS   = ['vocals','drums','bass']   # use 3 stems

## 1️⃣ Load Pretrained AST Feature Extractor
The `ASTFeatureExtractor` handles all audio → model input conversion for us.

In [ ]:
MODEL_NAME = 'MIT/ast-finetuned-audioset-10-10-0.4593'
print(f'Loading feature extractor from {MODEL_NAME}...')
feature_extractor = ASTFeatureExtractor.from_pretrained(MODEL_NAME)
print('✅ Feature extractor loaded')
print(f'   Sample rate : {feature_extractor.sampling_rate}')
print(f'   Max length  : {feature_extractor.max_length}')

In [ ]:
CONFIG = {
    'model_name'  : MODEL_NAME,
    'batch_size'  : 16,         # smaller because AST is large
    'epochs'      : 20,
    'lr'          : 1e-4,       # small LR for fine-tuning
    'weight_decay': 1e-4,
    'warmup_steps': 100,
    'duration'    : 30,
    'use_stems'   : USE_STEMS,
    'model'       : 'AST_finetuned',
}

TARGET_SR = feature_extractor.sampling_rate
print(f'Target sample rate: {TARGET_SR} Hz')

## 2️⃣ Dataset with AST Preprocessing

In [ ]:
class ASTDataset(Dataset):
    """
    Loads audio files and processes them using the AST feature extractor.
    Returns input_values (mel-filterbank features) ready for the transformer.
    """
    def __init__(self, file_list, label_list=None, augment=False):
        self.files   = file_list
        self.labels  = label_list
        self.augment = augment

    def __len__(self): return len(self.files)

    def __getitem__(self, idx):
        path = self.files[idx]
        try:
            y, sr = librosa.load(path, sr=TARGET_SR, duration=CONFIG['duration'])
            if len(y) < TARGET_SR:
                y = np.zeros(TARGET_SR * CONFIG['duration'])

            # Optional: simple noise augmentation
            if self.augment and random.random() < 0.3:
                noise = np.random.randn(len(y)) * 0.005
                y = y + noise

        except:
            y = np.zeros(TARGET_SR * CONFIG['duration'])

        # Feature extractor returns a dict with 'input_values'
        inputs = feature_extractor(
            y,
            sampling_rate=TARGET_SR,
            return_tensors='pt',
            padding='max_length',
            max_length=feature_extractor.max_length,
        )
        item = {'input_values': inputs['input_values'].squeeze(0)}  # (1024, 128)
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

# Build file lists
all_files, all_labels = [], []
for genre in GENRES:
    for song in sorted(os.listdir(f'{STEMS_DIR}/{genre}')):
        for stem in USE_STEMS:
            path = f'{STEMS_DIR}/{genre}/{song}/{stem}.wav'
            if os.path.exists(path):
                all_files.append(path)
                all_labels.append(genre)

le = LabelEncoder()
labels_enc = le.fit_transform(all_labels)

X_tr, X_val, y_tr, y_val = train_test_split(
    all_files, labels_enc, test_size=0.2, stratify=labels_enc, random_state=42)

print(f'Train: {len(X_tr)}, Val: {len(X_val)}')

train_loader = DataLoader(ASTDataset(X_tr, y_tr, augment=True),
                          batch_size=CONFIG['batch_size'], shuffle=True, num_workers=2)
val_loader   = DataLoader(ASTDataset(X_val, y_val, augment=False),
                          batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2)